In [ ]:
df = spark.read.table("transport_vic_dev.01_bronze.bronze_vline_trip_updates")
display(df.head())

Row(path='abfss://lakehouse@transportvicdatalake.dfs.core.windows.net/landing/vline_trip_updates/date=20260707/vline_tu_20260707T013500Z.pb', modificationTime=datetime.datetime(2026, 7, 7, 11, 35), length=10674, content=b'\n\r\n\x032.0\x10\x00\x18\xc2\xaf\xb1\xd2\x06\x12v\n\x1a01-SWL--6-T0-8080|20260707\x1aX\n>\n\x1101-SWL--6-T0-8080\x12\x0806:52:00\x1a\x0820260707 \x00*\x13aus:vic:vic-01-SWL:\x12\x16\x08\x0f\x12\t\x08\x96\x04\x10\xae\xb1\xb1\xd2\x06"\x0520043(\x00\x12\xae\x01\n\x1a01-SWL--6-T0-8081|20260707\x1a\x8f\x01\n>\n\x1101-SWL--6-T0-8081\x12\x0807:38:00\x1a\x0820260707 \x00*\x13aus:vic:vic-01-SWL:\x12\x1b\x08\r\x12\x06\x10\xac\xa3\xb1\xd2\x06\x1a\x06\x10\xac\xa3\xb1\xd2\x06"\x0520336(\x01\x12\x1b\x08\x0e\x12\x06\x10\xbc\xb0\xb1\xd2\x06\x1a\x06\x10\xbc\xb0\xb1\xd2\x06"\x0520318(\x01\x12\x13\x08\x0f\x12\x06\x10\xc0\xc6\xb1\xd2\x06"\x0520347(\x01\x12~\n\x1a01-BDE--6-T0-8461|20260707\x1a`\n>\n\x1101-BDE--6-T0-8461\x12\x0807:40:00\x1a\x0820260707 \x00*\x13aus:vic:vic-01-BDE:\x12\x1e

In [3]:
row_count = df.count()
print(f"Total Rows: {row_count}")

Total Rows: 21


In [5]:
%sql

SELECT COUNT(*)          AS files,
       MIN(length)/1024  AS min_kb,
       AVG(length)/1024  AS avg_kb,
       MAX(length)/1024  AS max_kb
FROM   transport_vic_dev.`01_bronze`.bronze_vline_trip_updates;

,files,min_kb,avg_kb,max_kb
0,21,9.066406,9.766602,10.423828


## Checking the number of files pending to be processed

In [6]:
base = "abfss://lakehouse@transportvicdatalake.dfs.core.windows.net/landing/vline_trip_updates/"

landing = (spark.read.format("binaryFile")
           .option("recursiveFileLookup", "true").option("pathGlobFilter", "*.pb")
           .load(base).select("path"))                     # .select(path) prunes the content bytes
bronze  = spark.table("transport_vic_dev.`01_bronze`.bronze_vline_trip_updates").select("path")

pending = landing.join(bronze, "path", "left_anti")
print(f"landing={landing.count()}  bronze={bronze.count()}  pending={pending.count()}")
pending.show(truncate=False)                                # the exact files awaiting the next run

landing=33  bronze=21  pending=12
+---------------------------------------------------------------------------------------------------------------------------------+
|path                                                                                                                             |
+---------------------------------------------------------------------------------------------------------------------------------+
|abfss://lakehouse@transportvicdatalake.dfs.core.windows.net/landing/vline_trip_updates/date=20260707/vline_tu_20260707T033500Z.pb|
|abfss://lakehouse@transportvicdatalake.dfs.core.windows.net/landing/vline_trip_updates/date=20260707/vline_tu_20260707T034000Z.pb|
|abfss://lakehouse@transportvicdatalake.dfs.core.windows.net/landing/vline_trip_updates/date=20260707/vline_tu_20260707T033000Z.pb|
|abfss://lakehouse@transportvicdatalake.dfs.core.windows.net/landing/vline_trip_updates/date=20260707/vline_tu_20260707T024500Z.pb|
|abfss://lakehouse@transportvicdatalake.df

In [9]:
from google.transit import gtfs_realtime_pb2
from google.protobuf.json_format import MessageToJson

rows = (spark.read.table("transport_vic_dev.`01_bronze`.bronze_vline_trip_updates")
        .select("path", "content").limit(5).collect())      # bytes come to local Python

for r in rows:
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(r["content"])
    print(r["path"])
    print(MessageToJson(feed, preserving_proto_field_name=True))

abfss://lakehouse@transportvicdatalake.dfs.core.windows.net/landing/vline_trip_updates/date=20260707/vline_tu_20260707T013500Z.pb
{
  "header": {
    "gtfs_realtime_version": "2.0",
    "incrementality": "FULL_DATASET",
    "timestamp": "1783388098"
  },
  "entity": [
    {
      "id": "01-SWL--6-T0-8080|20260707",
      "trip_update": {
        "trip": {
          "trip_id": "01-SWL--6-T0-8080",
          "start_time": "06:52:00",
          "start_date": "20260707",
          "schedule_relationship": "SCHEDULED",
          "route_id": "aus:vic:vic-01-SWL:"
        },
        "stop_time_update": [
          {
            "stop_sequence": 15,
            "arrival": {
              "delay": 534,
              "time": "1783388334"
            },
            "stop_id": "20043",
            "schedule_relationship": "SCHEDULED"
          }
        ]
      }
    },
    {
      "id": "01-SWL--6-T0-8081|20260707",
      "trip_update": {
        "trip": {
          "trip_id": "01-SWL--6-T0-8081"